# E-commerce Fulfilment & Delivery Performance Analysis

## Notebook 02: Data Cleaning and Preparation

This notebook focuses on cleaning and preparing the raw Olist e-commerce data for analysis.

The purpose of this notebook is to:

- Load the raw CSV files
- Convert important date columns into proper datetime format
- Focus on delivered orders for delivery performance analysis
- Create delivery-related calculated fields
- Prepare clean datasets for later SQL, Power BI and exploratory analysis

The first notebook focused on understanding the raw data. This notebook starts preparing the data for analysis.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Define project folders

raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")

# Check that both folders exist
print("Raw data folder exists:", raw_data_path.exists())
print("Processed data folder exists:", processed_data_path.exists())

Raw data folder exists: True
Processed data folder exists: True


## Load Raw Data

In this section, I will load the raw CSV files again.

Although the files were already loaded in Notebook 01, each notebook should be able to run independently. This means Notebook 02 should not depend on Notebook 01 being open or already run.

In [3]:
# Load raw datasets

customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")
order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(raw_data_path / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")
products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(raw_data_path / "product_category_name_translation.csv")

print("Raw datasets loaded successfully.")

Raw datasets loaded successfully.


## Initial Orders Table Check

The orders table is the main table for delivery performance analysis because it contains:

- Order ID
- Customer ID
- Order status
- Purchase timestamp
- Carrier delivery date
- Customer delivery date
- Estimated delivery date

Before cleaning, I will inspect the structure of the orders table.

In [4]:
# Preview the orders table

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
# Check columns and data types in the orders table

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


## Convert Date Columns

The raw orders table contains several date columns.

At first, pandas reads these columns as text. To calculate delivery days and late delivery, these columns need to be converted into datetime format.

Datetime format allows Python to calculate the difference between dates.

In [6]:
# Create a copy of the orders table for cleaning

orders_clean = orders.copy()

# List of date columns to convert
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert date columns to datetime format
for column in date_columns:
    orders_clean[column] = pd.to_datetime(orders_clean[column], errors="coerce")

# Check data types after conversion
orders_clean[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Filter to Delivered Orders

For delivery performance analysis, I will focus on orders with an order status of delivered.

This is because late delivery can only be measured properly when the actual customer delivery date is available.

Cancelled, unavailable, shipped or invoiced orders may be useful for other analysis later, but they are not the main focus for delivery performance KPIs.

In [7]:
# Check order status counts

orders_clean["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
# Filter to delivered orders only

delivered_orders = orders_clean[orders_clean["order_status"] == "delivered"].copy()

# Check the number of delivered orders
print("Total orders:", len(orders_clean))
print("Delivered orders:", len(delivered_orders))

Total orders: 99441
Delivered orders: 96478


## Create Delivery Performance Fields

In this section, I will create new calculated fields for delivery analysis.

These fields will help answer key business questions such as:

- How long did orders take to reach customers?
- Were orders delivered before or after the estimated delivery date?
- How many days late were late orders?
- Which orders were delivered late?

In [9]:
# Create delivery performance fields

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"] 
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["estimated_delivery_days"] = (
    delivered_orders["order_estimated_delivery_date"] 
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"] 
    - delivered_orders["order_estimated_delivery_date"]
).dt.days

delivered_orders["is_late_delivery"] = delivered_orders["delay_days"] > 0

# Preview the new fields
delivered_orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "estimated_delivery_days",
        "delay_days",
        "is_late_delivery"
    ]
].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delay_days,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,15,-8.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,19,-6.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,26,-18.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,26,-13.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,12,-10.0,False


In [10]:
# Summary statistics for delivery performance fields

delivered_orders[
    [
        "delivery_days",
        "estimated_delivery_days",
        "delay_days"
    ]
].describe()

,delivery_days,estimated_delivery_days,delay_days
count,96470.000000,96478.000000,96470.000000
mean,12.093604,23.372759,-11.875889
std,9.551380,8.758137,10.182105
min,0.000000,2.000000,-147.000000
25%,6.000000,18.000000,-17.000000
50%,10.000000,23.000000,-12.000000
75%,15.000000,28.000000,-7.000000
max,209.000000,155.000000,188.000000


In [11]:
# Late delivery count and rate

late_delivery_count = delivered_orders["is_late_delivery"].sum()
delivered_order_count = len(delivered_orders)
late_delivery_rate = late_delivery_count / delivered_order_count

print("Delivered orders:", delivered_order_count)
print("Late deliveries:", late_delivery_count)
print("Late delivery rate:", round(late_delivery_rate * 100, 2), "%")

Delivered orders: 96478
Late deliveries: 6534
Late delivery rate: 6.77 %


## Export Cleaned Orders Dataset

The cleaned delivered orders dataset will be saved into the processed data folder.

This processed file will be used later for SQL analysis, Power BI dashboards and further Python analysis.

In [12]:
# Export cleaned delivered orders dataset

output_file = processed_data_path / "cleaned_delivered_orders.csv"

delivered_orders.to_csv(output_file, index=False)

print("Cleaned delivered orders file saved to:")
print(output_file)

Cleaned delivered orders file saved to:
..\data\processed\cleaned_delivered_orders.csv


In [13]:
# Check that the processed file was created

output_file.exists()

True